# ДЗ 14. NumPy: векторизация, broadcasting, оси, маски 🔢
Решите задачи в Google Colab и сдайте через GitHub: папка `hw14` в репозитории `cyberformula3-homework`.

- **[База]** — обязательно для всех ·
- **[Челлендж]** — сколько получится · 
- **[Про]** — для тех, кому легко · 

Каждый ответ — **число + одна фраза**: что это значит для магазина.
Циклы `for` по заказам для расчётов не используем — только массивы и маски. Цикл можно там, где это сказано.

**Проверь себя перед сдачей:**
□ **Среда выполнения → Перезапустить и выполнить все** — ноутбук проходит без ошибок
□ Ответы совпадают с «Примером проверки», где он есть

In [7]:
import numpy as np
# проверка: вставьте сюда строки 1–5 и запустите


## Загрузка данных — запустите как есть

In [10]:
# Загрузка: в Colab нажмите «Выбрать файлы» и укажите orders.csv из LMS.
import os
if not os.path.exists("orders_2.csv"):
    try:
        from google.colab import files
        files.upload()
    except ImportError:
        print("Положите orders.csv в одну папку с ноутбуком")
FILE = "orders_2.csv"
print("Работаем с файлом:", FILE)

Работаем с файлом: orders_2.csv


In [12]:
import csv
import json
import numpy as np

with open(FILE, encoding="utf-8") as f:
    rows = list(csv.DictReader(f))

price, qty, city, category, status, month, product = [], [], [], [], [], [], []
for r in rows:
    price.append(int(r["price"]))
    qty.append(int(r["qty"]))
    city.append(r["city"])
    category.append(r["category"])
    status.append(r["status"])
    month.append(int(r["date"].split("-")[1]))
    product.append(r["product"])

price = np.array(price)
qty = np.array(qty)
city = np.array(city)
category = np.array(category)
status = np.array(status)
month = np.array(month)
product = np.array(product)

revenue = price * qty
paid = status == "оплачен"
print(revenue.shape, revenue[:5])

(500,) [198000  24000 159000 189000 129000]


## [База]

### Задача 1. Цены выросли на 5%
Магазин поднимает все цены на 5%. Посчитайте новые суммы заказов **одной строкой**, без цикла.
На сколько тенге вырастет общая выручка?

**Пример проверки:** выручка вырастет на `4 625 325 ₸`.

In [16]:
# TODO
print(revenue*1.05)
print(f"Выручка вырастет на {sum(revenue)*0.05}")

[ 207900.   25200.  166950.  198450.  135450.  334950.  103950.  103950.
   93450.  334950.   12600.  198450.   26250.  334950.  576450.   26250.
   78750.   12600.   78750.  471450.   14700.   37800.  471450.  471450.
  103950.   12600.   12600.  334950.  135450.  103950.  303450.  166950.
   12600.  198450.  606900.  576450.   93450.  576450.  103950.   26250.
  198450.    7875.   12600.  334950.  198450.  334950.  250950.  303450.
   37800.  501900.   25200.  303450.  135450.  166950.   23625.   44100.
  166950.   26250.  334950.  334950.   37800.  576450.  576450.  303450.
  198450.   14700.  471450.  135450.    7875.   12600.  303450.   78750.
  334950.  198450.  198450. 1152900.  303450.  334950.   12600.  471450.
  103950.  250950.  135450.  103950.  576450.   78750.  576450.  198450.
  103950.  250950.    7875.  166950.  198450.  250950.  103950.   14700.
   44100.  471450.  396900.  166950.   12600.    7875.   14700.  303450.
   14700.  334950.  334950.  334950.  207900.  1039

### Задача 2. Смартфоны
С помощью маски по `category` найдите:
- сколько заказов в категории «Смартфоны»;
- на какую сумму;
- какая это доля от всех заказов.

**Пример проверки:** `131` заказ на `30 689 000 ₸`, доля `26.2%`.

In [22]:
# TODO
smarts = category == "Смартфоны"
# print(smarts)
print(f"В категории смартфоны {sum(smarts == True)} заказ(-а,-ов) на {sum(revenue[smarts == True])}")

В категории смартфоны 131 заказ(-а,-ов) на 30689000


### Задача 3. Два условия
Сколько **оплаченных** заказов из **Шымкента** и на какую сумму? Не забудьте скобки вокруг каждого условия.

**Пример проверки:** `71` заказ на `12 852 500 ₸`.

In [35]:
# TODO
print(f"оплаченных заказов из Шимкента {sum((paid == True) & (city == 'Шымкент'))} на сумму {revenue[(paid == True) & (city == 'Шымкент')].sum()}")

оплаченных заказов из Шимкента 71 на сумму 12852500


### Задача 4. Неделя курьера
Первые 30 заказов — это 6 дней по 5 заказов: `w = revenue[:30].reshape(6, 5)`.
1. Сумма за каждый день — какую `axis` взять? Сколько чисел должно получиться?
2. В какой день (номер с 1) сумма самая большая?
3. Самый дорогой заказ **каждого** дня.

**Пример проверки:** лучший — **3-й** день, `1 094 000 ₸`; самые дорогие по дням — `[198000 319000 549000 449000 449000 319000]`.

In [42]:
w = revenue[:30].reshape(6, 5)
# TODO
print(f"нужно взять axis = 1 для подсчета по горизонтали. Сумма за каждый день {w.sum(axis=1)}. Должно получится 6 чисел.")
print(f"лучший {w.sum(axis=1).argmax()+1}-й день. Сумма {w.sum(axis=1).max()}")
print(f"Самые дорогие по дням {w.max(axis=1)}")

нужно взять axis = 1 для подсчета по горизонтали. Сумма за каждый день [ 699000  925000 1094000  636000 1047000  571000]. Должно получится 6 чисел.
лучший 3-й день. Сумма 1094000
Самые дорогие по дням [198000 319000 549000 449000 449000 319000]


## [Челлендж]

### Задача 5. Заказы в трёх валютах
Покупатели из-за рубежа просят показать суммы заказов в своей валюте.
Курсы Нацбанка РК на 19 ноября 2026: 1 USD = 444.16 ₸, 1 EUR = 487.29 ₸, 1 RUB = 5.40 ₸.

1. Возьмите суммы **первых 5 заказов** из `revenue` и получите **одной операцией** таблицу «заказ × валюта» формы `(5, 3)`, округлите до 2 знаков.
   Какую форму нужно дать суммам заказов, чтобы сработал broadcasting?
2. Выведите рядом товар каждого из этих 5 заказов — массив `product` уже собран в ячейке загрузки.


**Пример проверки:** форма суммам — `(5, 1)`; первый заказ (Xiaomi Redmi Note 13, 198 000 ₸) — `[445.79  406.33  36666.67]`.

In [101]:
# TODO

# print(f"суммам заказов нужно дать форму {revenue[:5].shape} для срабатывания Broadcasting и получения массива валют, {np.round(revenue[:5].reshape(1,5)/np.array([444.16, 487.29, 5.40]).reshape(3,1),2)}")
print(f"{product[:5].reshape(1,5)} + {revenue[:5].reshape(1,5)}  - {np.round(revenue[:5].reshape(5,1)/np.array([444.16, 487.29, 5.40]).reshape(1,3),2)[:5]}")

[['Xiaomi Redmi Note 13' 'Клавиатура Logitech K120'
  'Samsung Galaxy Tab A9' 'Samsung Galaxy A55' 'Монитор LG 27 дюймов']] + [[198000  24000 159000 189000 129000]]  - [[  445.79   406.33 36666.67]
 [   54.03    49.25  4444.44]
 [  357.98   326.29 29444.44]
 [  425.52   387.86 35000.  ]
 [  290.44   264.73 23888.89]]


### Задача 6. Категории
Для **каждой** категории из `np.unique(category)` выведите число заказов, выручку и средний чек (цикл по категориям можно).
Какая категория приносит больше всего денег? А у какой самый большой средний чек?

**Пример проверки:** и по выручке, и по среднему чеку лидируют **Ноутбуки** — `40 992 000 ₸`, средний чек ≈ `422 598 ₸`.

In [125]:
# TODO
cat = np.unique(category)
for i in cat:
    print(f'в категории {i} {len(revenue[i == category])} заказов на сумму {sum(revenue[i == category])}, средний чек {revenue[i == category].mean()} ')


в категории Аксессуары 171 заказов на сумму 4514500, средний чек 26400.584795321636 
в категории Мониторы 50 заказов на сумму 6046000, средний чек 120920.0 
в категории Ноутбуки 97 заказов на сумму 40992000, средний чек 422597.9381443299 
в категории Планшеты 51 заказов на сумму 10265000, средний чек 201274.50980392157 
в категории Смартфоны 131 заказов на сумму 30689000, средний чек 234267.17557251907 


### Задача 7. Бесплатная доставка
Доставка бесплатна, если сумма заказа от 50 000 ₸. С помощью `np.where` подпишите каждый заказ: `"бесплатно"` или `"платно"`.
Сколько заказов получили бесплатную доставку?

**Пример проверки:** `354`.

In [130]:
# TODO
size = np.where(revenue >= 50_000, "бесплатно", "платно")
print(f'бесплатных {(size == "бесплатно").sum()}, платных {(size == "платно").sum()}')

бесплатных 354, платных 146


## [Про]

### Задача 8. Таблица «категория × месяц»
Постройте таблицу **числа заказов** формы `(5, 8)`: строки — категории (`np.unique(category)`), столбцы — месяцы (`np.unique(month)`).
Двойной цикл по категориям и месяцам можно, внутри — маска из двух условий.

1. Проверьте: сумма всей таблицы должна быть равна числу заказов.
2. В каком месяце было больше всего заказов смартфонов?

**Пример проверки:** сумма таблицы `500`; больше всего смартфонов — в **сентябре (9)**, `22` заказа.

In [ ]:
# TODO

### Задача 9. Доли категорий по месяцам
Разделите таблицу на итог по **каждому месяцу** — получится доля каждой категории внутри месяца.
Это broadcasting: таблица `(5, 8)` делится на строку `(8,)`. Какая `axis` даёт итог по месяцам?

1. Проверьте: сумма каждого столбца после деления равна `1`.
2. Какая доля заказов в **августе** пришлась на аксессуары?
3. Сохраните доли в `shares.json`: `{"категории": [...], "месяцы": [...], "доли": [[...], ...]}`, доли округлите до 2 знаков. Не забудьте `.tolist()`.

**Пример проверки:** в августе аксессуары — `40.4%` заказов.

In [ ]:
# TODO